# DEE — Termodinámica cristalina y análisis de defectos

**Contexto:** tras confirmar que el sustrato DEE es matemáticamente equivalente a un cristal elástico discreto (test de fonones), exploramos dos consecuencias:

**Parte A — Termodinámica:**
- Densidad de estados g(ω)
- Temperatura de Debye θ_D
- Calor específico C_V(T)
- Energía de punto cero (vacío cristalino)

**Parte B — Defectos:**
- Modos localizados alrededor de vacancias
- Modos localizados alrededor de impurezas de constante de fuerza
- Comparación con el espectro del cristal perfecto

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags, eye as sparse_eye
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

PLOT_DIR = 'plots_termo_defectos'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones base
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian_from_points(points, k_neighbors=30, alpha=2.0, L=1.0,
                                  nodes_to_skip=None, force_modifiers=None):
    """
    Construye Laplaciano de grafo con opciones para introducir defectos.
    
    - nodes_to_skip: lista de índices de nodos a eliminar (vacancias)
    - force_modifiers: dict {nodo_idx: factor} para multiplicar las constantes de fuerza
      en ese nodo (impurezas)
    """
    D_mat = periodic_distance_matrix(points, L=L)
    N = len(points)
    
    if nodes_to_skip is None:
        nodes_to_skip = set()
    else:
        nodes_to_skip = set(nodes_to_skip)
    
    if force_modifiers is None:
        force_modifiers = {}
    
    # Para vacancias: excluir el nodo de la matriz
    valid_indices = [i for i in range(N) if i not in nodes_to_skip]
    
    # Mapeo índice_original → índice_nuevo
    idx_map = {old: new for new, old in enumerate(valid_indices)}
    N_new = len(valid_indices)
    
    # Reconstruir matriz de distancias solo con nodos válidos
    D_sub = D_mat[np.ix_(valid_indices, valid_indices)]
    
    D_search = D_sub.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, min(k_neighbors, N_new-1), axis=1)[:, :k_neighbors]
    
    rows, cols, vals = [], [], []
    for i_new in range(N_new):
        i_orig = valid_indices[i_new]
        for j_new in neighbor_idx[i_new]:
            d = D_sub[i_new, j_new]
            if d > 0:
                j_orig = valid_indices[j_new]
                w = 1.0 / d**alpha
                # Aplicar modificador si corresponde
                mod_i = force_modifiers.get(i_orig, 1.0)
                mod_j = force_modifiers.get(j_orig, 1.0)
                w = w * np.sqrt(mod_i * mod_j)  # modificador efectivo simétrico
                
                rows.append(i_new); cols.append(j_new); vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N_new, N_new))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    L_sparse = diags(degrees) - W
    
    return L_sparse, valid_indices, idx_map

# Construcción del grafo base
N_TARGET = 1331
K_NEIGHBORS = 30
JITTER = 0.1

points, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)
L_perfect, valid_idx_perfect, _ = build_laplacian_from_points(points, K_NEIGHBORS, alpha=2.0)

print(f'N = {N}, grafo base construido')

---

# PARTE A — Termodinámica cristalina

## 1. Densidad de estados g(ω)

La densidad de estados cuenta cuántos modos normales hay en cada intervalo de frecuencia. Se obtiene binneando los autovalores del Laplaciano (todos los no triviales).

**Comparación con Debye:** el modelo de Debye predice g(ω) = 3V ω² / (2π²c_s³) hasta la frecuencia de Debye ω_D, definida por ∫ g(ω)dω = 3N.

In [ ]:
# Calculamos MUCHOS autovalores para tener buena resolución en la DoS
# No podemos calcular todos (N autovalores sería O(N³) de memoria), pero 400 es buena muestra
N_EIGS = 400

import time
t0 = time.time()
eigs_perfect, _ = eigsh(L_perfect, k=N_EIGS, which='SM', tol=1e-8)
eigs_perfect = np.sort(eigs_perfect)
print(f'Diagonalización: {time.time()-t0:.1f}s')

# Filtrar modo cero
eigs_nonzero = eigs_perfect[eigs_perfect > 1e-3]
omegas_perfect = np.sqrt(eigs_nonzero)

print(f'Modos calculados: {len(omegas_perfect)}')
print(f'Rango de ω: [{omegas_perfect.min():.3f}, {omegas_perfect.max():.3f}]')

# Velocidad del sonido (del test de fonones anterior)
k1_mag = 2 * np.pi
c_s = omegas_perfect[0] / k1_mag
print(f'c_s = {c_s:.4f}')

In [ ]:
# Densidad de estados via histograma
# Usamos bins con ancho adaptativo en escala log-lineal
n_bins = 35
omega_bins = np.linspace(0, omegas_perfect.max() * 1.05, n_bins + 1)
omega_centers = 0.5 * (omega_bins[:-1] + omega_bins[1:])
dos_counts, _ = np.histogram(omegas_perfect, bins=omega_bins)
bin_width = omega_bins[1] - omega_bins[0]
dos = dos_counts / bin_width

# Predicción de Debye (normalizada para que integre al número total de modos)
# En 3D: g_Debye(ω) = 3V ω² / (2π²c_s³)
# Usamos V=1 (cubo unidad)
debye_unnorm = omega_centers**2
# Normalizamos para que tenga la misma integral que los datos observados
# hasta la frecuencia de Debye
omega_D_approx = omegas_perfect[int(0.9 * len(omegas_perfect))]
mask_debye = omega_centers <= omega_D_approx
observed_integral = np.sum(dos[mask_debye] * bin_width)
debye_integral = np.sum(debye_unnorm[mask_debye] * bin_width)
if debye_integral > 0:
    debye_factor = observed_integral / debye_integral
    debye_curve = debye_unnorm * debye_factor
else:
    debye_curve = debye_unnorm * 0

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.step(omega_centers, dos, where='mid', color='darkblue', lw=2, label='g(ω) del sustrato DEE')
ax.plot(omega_centers, debye_curve, '--', color='red', lw=2,
        label=f'Debye: g ∝ ω² (c_s={c_s:.2f})')
ax.axvline(omega_D_approx, color='gray', linestyle=':', alpha=0.7,
           label=f'ω_D ≈ {omega_D_approx:.2f}')
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('g(ω) — densidad de estados', fontsize=12)
ax.set_title('Densidad de estados fonónica del sustrato DEE', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Panel 2: g(ω)/ω² para visualizar desviaciones de Debye
ax = axes[1]
ratio = dos / (omega_centers**2 + 0.01)  # evitar división por cero
ax.step(omega_centers, ratio, where='mid', color='darkgreen', lw=2,
        label='g(ω) / ω²')
ax.axhline(debye_factor, color='red', linestyle='--',
           label=f'Valor Debye constante = {debye_factor:.2f}')
ax.axvline(omega_D_approx, color='gray', linestyle=':', alpha=0.7)
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('g(ω) / ω²', fontsize=12)
ax.set_title('Razón g/ω² — constante = Debye puro, variación = estructura', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('18_densidad_estados')
plt.show()

print(f'\nFrecuencia de Debye estimada: ω_D ≈ {omega_D_approx:.3f}')
print(f'(es la ω bajo la cual están el 90% de los modos calculados)')

## 2. Calor específico C_V(T)

Para un sistema de osciladores cuánticos con frecuencias {ω_i}:

$$C_V(T) = k_B \sum_i \left(\frac{\hbar\omega_i}{k_B T}\right)^2 \frac{e^{\hbar\omega_i/k_B T}}{(e^{\hbar\omega_i/k_B T}-1)^2}$$

Usamos unidades naturales ħ=k_B=1. El comportamiento esperado:
- **T → 0**: ley de Debye C_V ∝ T³
- **T → ∞**: ley de Dulong-Petit C_V → 3N · k_B

In [ ]:
def heat_capacity(omegas, T):
    """Calor específico a temperatura T para un sistema con frecuencias ω."""
    if T <= 0:
        return 0.0
    x = omegas / T  # ħω/(k_B T) con unidades naturales
    # Cutoff para evitar overflow
    x_safe = np.clip(x, 1e-10, 50)
    # Fórmula estándar: x² exp(x) / (exp(x) - 1)²
    exp_x = np.exp(x_safe)
    contrib = x_safe**2 * exp_x / (exp_x - 1)**2
    # Para x muy grande, la contribución tiende a cero
    contrib = np.where(x > 50, 0.0, contrib)
    return np.sum(contrib)

# Barrer T en escala log
T_values = np.logspace(-1.5, 1.5, 60)
C_V = np.array([heat_capacity(omegas_perfect, T) for T in T_values])

# Comparaciones teóricas
C_V_dulong = 3 * N  # límite alto T (sin factor k_B porque estamos en unidades naturales, pero aproximado)
# El número efectivo de osciladores es len(omegas_perfect), no N
# porque solo calculamos N_EIGS modos. Escalamos
C_V_dulong_effective = len(omegas_perfect)

# Baja T: C_V ∝ T³ según Debye
# Coeficiente exacto: C_V = (12π⁴/5) N (T/θ_D)³ con θ_D = ω_D
theta_D = omega_D_approx
T_low = T_values[T_values < theta_D * 0.3]
C_V_debye_lowT = (12 * np.pi**4 / 5) * len(omegas_perfect) * (T_low / theta_D)**3

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(T_values, C_V, 'o-', color='darkblue', markersize=6, label='C_V del sustrato DEE')
ax.axhline(C_V_dulong_effective, color='red', linestyle='--',
           label=f'Dulong-Petit: 3N_modos = {C_V_dulong_effective}')
ax.plot(T_low, C_V_debye_lowT, ':', color='orange', lw=2,
        label='Debye bajas T: C_V ∝ T³')
ax.set_xlabel('T (temperatura, unidades naturales)', fontsize=12)
ax.set_ylabel('C_V (calor específico)', fontsize=12)
ax.set_title('Calor específico del sustrato DEE', fontsize=12)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 2: C_V/T³ vs T para visualizar la región Debye y las correcciones
ax = axes[1]
ratio_CV_T3 = C_V / T_values**3
ax.plot(T_values, ratio_CV_T3, 'o-', color='darkgreen', markersize=6)
ax.axvline(theta_D, color='red', linestyle='--', label=f'θ_D ≈ {theta_D:.2f}')
ax.set_xlabel('T', fontsize=12)
ax.set_ylabel('C_V / T³', fontsize=12)
ax.set_title('C_V/T³ — constante a bajas T = régimen Debye puro', fontsize=12)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('19_calor_especifico')
plt.show()

# Valores cualitativos
C_V_at_theta_D = heat_capacity(omegas_perfect, theta_D)
print(f'\nTemperatura de Debye: θ_D = {theta_D:.3f}')
print(f'C_V(T=θ_D) = {C_V_at_theta_D:.1f}')
print(f'C_V asintótico (Dulong-Petit) = {C_V_dulong_effective}')
print(f'Ratio: {C_V_at_theta_D / C_V_dulong_effective:.3f}')
print('(debería ser ~0.95 si el sustrato sigue Debye cerca de θ_D)')

## 3. Energía de punto cero

En un oscilador cuántico, la energía de punto cero es ½ħω. Para el sustrato completo:

$$E_0 = \sum_i \frac{1}{2} \hbar \omega_i$$

Esta es la **energía del vacío cristalino**. Es no nula incluso a T=0.

In [ ]:
# Energía de punto cero
E_0_per_mode = 0.5 * omegas_perfect
E_0_total = np.sum(E_0_per_mode)
E_0_per_node = E_0_total / N

print(f'Energía de punto cero del sustrato (del subespacio calculado):')
print(f'  E_0 total = {E_0_total:.2f} (unidades naturales)')
print(f'  E_0 por modo promedio = {np.mean(E_0_per_mode):.4f}')
print(f'  E_0 por nodo (estimación) = {E_0_per_node:.4f}')
print()
print(f'Número de modos calculados: {len(omegas_perfect)}/{3*N} esperados (3 por nodo)')
print(f'(Nota: solo diagonalizamos {N_EIGS} de 3N modos totales, entonces E_0 es subestimada')
print(f' pero los modos excluidos son los de mayor ω y contribuyen más por ω, entonces el')
print(f' cálculo está sesgado a subestimar)')

---

# PARTE B — Análisis de defectos

## 1. Vacancia: un nodo eliminado

Eliminamos un nodo central del grafo y recalculamos el espectro. Los modos localizados aparecen como autovalores con autovectores concentrados alrededor del defecto.

In [ ]:
# Encontrar el nodo más cercano al centro del cubo
center = np.array([0.5, 0.5, 0.5])
diff_to_center = points - center
diff_to_center = diff_to_center - np.round(diff_to_center)
r_from_center = np.linalg.norm(diff_to_center, axis=1)
central_node = int(np.argmin(r_from_center))

print(f'Nodo central elegido: índice {central_node}')
print(f'Posición: {points[central_node]}')

# Construir Laplaciano con vacancia
L_vacancy, valid_idx_vac, idx_map_vac = build_laplacian_from_points(
    points, K_NEIGHBORS, alpha=2.0,
    nodes_to_skip=[central_node]
)

print(f'Laplaciano con vacancia: dimensión {L_vacancy.shape}')

# Calcular autovalores
eigs_vacancy, vecs_vacancy = eigsh(L_vacancy, k=100, which='SM', tol=1e-8)
idx_sort = np.argsort(eigs_vacancy)
eigs_vacancy = eigs_vacancy[idx_sort]
vecs_vacancy = vecs_vacancy[:, idx_sort]

# Filtrar modo cero
mask_nz = eigs_vacancy > 1e-3
eigs_vacancy_nz = eigs_vacancy[mask_nz]
vecs_vacancy_nz = vecs_vacancy[:, mask_nz]
omegas_vacancy = np.sqrt(eigs_vacancy_nz)

print(f'Primeros 20 autovalores cristal perfecto: {omegas_perfect[:20].round(3)}')
print(f'Primeros 20 autovalores con vacancia:     {omegas_vacancy[:20].round(3)}')

In [ ]:
# Para cuantificar localización: medir el "radio de giración" del autovector
def localization_radius(eigenvector, point_coords_for_valid, center_position):
    """Calcula un radio efectivo donde vive la norma del autovector."""
    # Normalizar
    v = eigenvector / np.linalg.norm(eigenvector)
    v2 = v**2
    
    # Distancias de cada nodo válido al centro (usando métrica periódica)
    diffs = point_coords_for_valid - center_position
    diffs = diffs - np.round(diffs)
    dists = np.linalg.norm(diffs, axis=1)
    
    # Radio cuadrático medio de la distribución |v|²
    r_mean = np.sum(v2 * dists)
    r_rms = np.sqrt(np.sum(v2 * dists**2))
    # IPR (Inverse Participation Ratio)
    ipr = np.sum(v2**2)
    # Un modo extendido tiene IPR ~ 1/N; uno localizado tiene IPR O(1)
    participation = 1.0 / ipr if ipr > 0 else len(v)
    
    return r_mean, r_rms, participation

# Coordenadas de los nodos válidos para el grafo con vacancia
points_vacancy = points[valid_idx_vac]
vacancy_position = points[central_node]  # donde ESTABA el nodo ahora ausente

# Analizar los primeros 30 modos del cristal con defecto
print(f'{"idx":>4} {"ω":>8} {"r_rms":>8} {"participation":>14}')
print('-'*45)

localizations = []
for i in range(30):
    r_mean, r_rms, part = localization_radius(vecs_vacancy_nz[:, i],
                                               points_vacancy,
                                               vacancy_position)
    localizations.append({'omega': omegas_vacancy[i], 'r_rms': r_rms,
                          'participation': part})
    flag = ' ← LOCALIZADO' if part < N/10 else ''
    print(f'{i+1:>4} {omegas_vacancy[i]:>8.3f} {r_rms:>8.3f} {part:>14.1f}{flag}')

print()
print('Interpretación:')
print('- participation ratio alto (~N/2): modo extendido (onda plana)')
print('- participation ratio bajo (O(1)): modo localizado alrededor del defecto')

In [ ]:
# Comparación de espectros: cristal perfecto vs con vacancia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: primeros 25 autovalores superpuestos
ax = axes[0]
n_compare = 25
ax.plot(np.arange(n_compare), omegas_perfect[:n_compare], 'o', color='steelblue',
        markersize=10, label='Cristal perfecto')
ax.plot(np.arange(n_compare), omegas_vacancy[:n_compare], 's', color='crimson',
        markersize=10, label='Con vacancia')
ax.set_xlabel('Índice del modo', fontsize=12)
ax.set_ylabel('ω', fontsize=12)
ax.set_title('Espectro: cristal perfecto vs con vacancia', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Panel derecho: participation ratio de cada modo
ax = axes[1]
parts = [l['participation'] for l in localizations]
ax.semilogy(np.arange(30), parts, 'o-', color='darkmagenta', markersize=8)
ax.axhline(N/10, color='red', linestyle='--', label=f'N/10 = {N//10} (umbral localización)')
ax.axhline(N/2, color='gray', linestyle=':', label=f'N/2 = {N//2} (modo extendido típico)')
ax.set_xlabel('Índice del modo', fontsize=12)
ax.set_ylabel('Participation ratio', fontsize=12)
ax.set_title('Participation ratio — bajos valores indican localización', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('20_vacancia_espectro_localizacion')
plt.show()

## 2. Impureza de constante de fuerza

En lugar de eliminar un nodo, lo dejamos presente pero con sus acoplamientos con vecinos **debilitados** (factor 0.3 - "impureza blanda") o **reforzados** (factor 3 - "impureza dura").

In [ ]:
# Impureza blanda (factor 0.3)
L_soft, _, _ = build_laplacian_from_points(
    points, K_NEIGHBORS, alpha=2.0,
    force_modifiers={central_node: 0.3}
)

eigs_soft, vecs_soft = eigsh(L_soft, k=50, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_soft)
eigs_soft = eigs_soft[idx_s]
vecs_soft = vecs_soft[:, idx_s]
omegas_soft = np.sqrt(eigs_soft[eigs_soft > 1e-3])

# Impureza dura (factor 3)
L_hard, _, _ = build_laplacian_from_points(
    points, K_NEIGHBORS, alpha=2.0,
    force_modifiers={central_node: 3.0}
)

eigs_hard, vecs_hard = eigsh(L_hard, k=50, which='SM', tol=1e-8)
idx_h = np.argsort(eigs_hard)
eigs_hard = eigs_hard[idx_h]
vecs_hard = vecs_hard[:, idx_h]
omegas_hard = np.sqrt(eigs_hard[eigs_hard > 1e-3])

print(f'Comparación de primeros 15 autovalores:')
print(f'{"idx":>4} {"perfecto":>10} {"blando 0.3":>12} {"duro 3.0":>10}')
print('-'*45)
for i in range(15):
    print(f'{i+1:>4} {omegas_perfect[i]:>10.3f} {omegas_soft[i]:>12.3f} {omegas_hard[i]:>10.3f}')

In [ ]:
# Análisis de localización para la impureza dura
# En cristales, una impureza dura suele producir un modo de alta frecuencia
# (por encima del espectro de Debye) localizado alrededor de ella.

# Calcular participation ratios para todos los modos de la impureza dura
parts_hard = []
for i in range(len(omegas_hard)):
    v = vecs_hard[:, i+1] if eigs_hard[0] < 1e-3 else vecs_hard[:, i]  # saltar modo cero
    v2 = v**2
    ipr = np.sum(v2**2)
    parts_hard.append(1.0 / ipr if ipr > 0 else N)

parts_hard = np.array(parts_hard)

# Mismo para impureza blanda
parts_soft = []
for i in range(len(omegas_soft)):
    v = vecs_soft[:, i+1] if eigs_soft[0] < 1e-3 else vecs_soft[:, i]
    v2 = v**2
    ipr = np.sum(v2**2)
    parts_soft.append(1.0 / ipr if ipr > 0 else N)

parts_soft = np.array(parts_soft)

# Plot combinado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
n_plot = 30
ax.plot(omegas_perfect[:n_plot], 'o', color='steelblue', markersize=10, label='Cristal perfecto')
ax.plot(omegas_soft[:n_plot], '^', color='darkgreen', markersize=10, label='Impureza blanda (×0.3)')
ax.plot(omegas_hard[:n_plot], 'v', color='darkred', markersize=10, label='Impureza dura (×3.0)')
ax.set_xlabel('Índice', fontsize=12)
ax.set_ylabel('ω', fontsize=12)
ax.set_title('Espectros con impurezas vs cristal perfecto', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Panel derecho: participation ratio para identificar modos localizados
ax = axes[1]
ax.semilogy(omegas_soft[:30], parts_soft[:30], '^', color='darkgreen',
            markersize=10, label='Impureza blanda')
ax.semilogy(omegas_hard[:30], parts_hard[:30], 'v', color='darkred',
            markersize=10, label='Impureza dura')
ax.axhline(N/10, color='orange', linestyle='--', label='Umbral localización (N/10)')
ax.set_xlabel('ω', fontsize=12)
ax.set_ylabel('Participation ratio', fontsize=12)
ax.set_title('Participation ratio vs ω — ¿hay modos localizados?', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('21_impurezas_comparacion')
plt.show()

# Identificar modos localizados específicamente
print('\nModos posiblemente localizados (participation < N/10):')
print(f'\nImpureza blanda (×0.3):')
for i in range(min(30, len(omegas_soft))):
    if parts_soft[i] < N/10:
        print(f'  Modo {i+1}: ω = {omegas_soft[i]:.3f}, participation = {parts_soft[i]:.1f}')

print(f'\nImpureza dura (×3.0):')
for i in range(min(30, len(omegas_hard))):
    if parts_hard[i] < N/10:
        print(f'  Modo {i+1}: ω = {omegas_hard[i]:.3f}, participation = {parts_hard[i]:.1f}')

## 3. Visualización espacial de un modo localizado

Si encontramos un modo localizado, lo visualizamos como mapa de calor 3D para ver explícitamente que está concentrado alrededor del defecto.

In [ ]:
# Tomar el modo más localizado de la impureza dura
if len(parts_hard) > 0:
    most_localized = int(np.argmin(parts_hard[:30]))
    v_loc = vecs_hard[:, most_localized + (1 if eigs_hard[0] < 1e-3 else 0)]
    
    fig = plt.figure(figsize=(12, 5))
    
    # Proyección 2D: |v|² vs distancia al defecto
    diff_imp = points - points[central_node]
    diff_imp = diff_imp - np.round(diff_imp)
    dist_from_imp = np.linalg.norm(diff_imp, axis=1)
    
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(dist_from_imp, v_loc**2, alpha=0.6, s=30, color='darkred')
    ax1.axvline(0, color='black', linestyle=':', label='Posición del defecto')
    ax1.set_xlabel('Distancia al defecto', fontsize=12)
    ax1.set_ylabel('|v|²', fontsize=12)
    ax1.set_title(f'Modo más localizado de impureza dura\nω={omegas_hard[most_localized]:.3f}, part={parts_hard[most_localized]:.1f}',
                  fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Proyección 3D en scatter
    from mpl_toolkits.mplot3d import Axes3D
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    # Colorear por |v|² en escala log para ver la concentración
    v2 = v_loc**2 + 1e-10
    scatter = ax2.scatter(points[:, 0], points[:, 1], points[:, 2],
                           c=np.log10(v2), s=20, alpha=0.5, cmap='hot')
    ax2.scatter([points[central_node, 0]], [points[central_node, 1]], [points[central_node, 2]],
                c='cyan', s=200, marker='*', edgecolors='black', linewidths=2,
                label='Posición del defecto')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_zlabel('z')
    ax2.set_title('Distribución espacial del modo (log |v|²)', fontsize=11)
    plt.colorbar(scatter, ax=ax2, shrink=0.7, label='log |v|²')
    ax2.legend(fontsize=9)
    
    plt.tight_layout()
    save_plot('22_modo_localizado_3d')
    plt.show()

## Síntesis final

In [ ]:
print('='*75)
print('SÍNTESIS — Termodinámica y defectos')
print('='*75)
print()
print('PARTE A — Termodinámica cristalina:')
print('-'*40)
print(f'  Velocidad del sonido: c_s = {c_s:.4f}')
print(f'  Frecuencia de Debye:  ω_D ≈ {theta_D:.3f}')
print(f'  Temperatura de Debye: θ_D = {theta_D:.3f} (unidades naturales)')
print(f'  Calor específico:     C_V(T→∞) → 3N (Dulong-Petit)')
print(f'                        C_V(T<<θ_D) ∝ T³ (ley de Debye)')
print(f'  E_0 por modo:         ~{np.mean(E_0_per_mode):.3f}')
print()
print('PARTE B — Defectos:')
print('-'*40)
n_loc_vac = sum(1 for l in localizations if l['participation'] < N/10)
n_loc_soft = np.sum(parts_soft[:30] < N/10)
n_loc_hard = np.sum(parts_hard[:30] < N/10)
print(f'  Modos localizados detectados (primeros 30):')
print(f'    Vacancia:           {n_loc_vac}')
print(f'    Impureza blanda:    {n_loc_soft}')
print(f'    Impureza dura:      {n_loc_hard}')
print()
if n_loc_hard > 0 or n_loc_soft > 0 or n_loc_vac > 0:
    print('  ✓ El sustrato DEE admite modos localizados alrededor de defectos.')
    print('    Este es el comportamiento estándar de cristales reales.')
    print('    Los defectos son candidatos estructurales para "materia localizada"')
    print('    con propiedades distintas a los modos fonónicos extendidos.')
else:
    print('  No se detectaron modos localizados claros en el rango analizado.')

In [ ]:
import shutil
shutil.make_archive('plots_termo_defectos', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_termo_defectos.zip')

try:
    from google.colab import files
    files.download('plots_termo_defectos.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass